# **Análisis estadístico en Python**

In [9]:
pip install duckdb pandas pyreadstat numpy scikit-learn statsmodels plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
import duckdb
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px

# ==============================================================================
# 1. CARGA DEL ARCHIVO .SAV Y REGISTRO EN DUCKDB
# ==============================================================================
os.chdir(r"C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025")
df = pd.read_spss("EH2025_Discriminacion.sav")

# Convertir columnas tipo 'category' de SPSS a texto plano para compatibilidad total con DuckDB
df_clean = df.copy()
for col in df_clean.select_dtypes(include=['category']).columns:
    df_clean[col] = df_clean[col].astype(str)

con = duckdb.connect(database=':memory:')
con.register('raw', df_clean)

# Verificación rápida de cómo vienen etiquetadas las categorías en el .sav
print("--- MUESTRA DE ETIQUETAS EN EH2025_Discriminacion.sav ---")
print(con.execute("SELECT DISTINCT depto, area, s09a_01a, s09b_01, s09b_02a FROM raw LIMIT 5").df())

# ==============================================================================
# 2. ETL E INGENIERÍA DE VARIABLES EN DUCKDB (SOPORTA CÓDIGOS Y ETIQUETAS SPSS)
# ==============================================================================
query_etl = """
CREATE OR REPLACE TABLE base_analitica AS
SELECT 
    folio,
    nro,
    -- Normalización de Departamento (soporta código '1'..'9' o etiqueta '1. Chuquisaca' / 'Chuquisaca')
    CASE 
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%chuquisaca%' OR TRIM(CAST(depto AS VARCHAR)) = '1' THEN 'Chuquisaca'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%la paz%'     OR TRIM(CAST(depto AS VARCHAR)) = '2' THEN 'La Paz'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%cochabamba%' OR TRIM(CAST(depto AS VARCHAR)) = '3' THEN 'Cochabamba'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%oruro%'      OR TRIM(CAST(depto AS VARCHAR)) = '4' THEN 'Oruro'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%potos%'      OR TRIM(CAST(depto AS VARCHAR)) = '5' THEN 'Potosí'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%tarija%'     OR TRIM(CAST(depto AS VARCHAR)) = '6' THEN 'Tarija'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%santa cruz%' OR TRIM(CAST(depto AS VARCHAR)) = '7' THEN 'Santa Cruz'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%beni%'       OR TRIM(CAST(depto AS VARCHAR)) = '8' THEN 'Beni'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%pando%'      OR TRIM(CAST(depto AS VARCHAR)) = '9' THEN 'Pando'
        ELSE CAST(depto AS VARCHAR)
    END AS departamento,

    -- Normalización de Área Geográfica
    CASE 
        WHEN LOWER(CAST(area AS VARCHAR)) LIKE '%urban%' OR TRIM(CAST(area AS VARCHAR)) = '1' THEN 'Urbana'
        ELSE 'Rural' 
    END AS area_geo,

    CAST(estrato AS VARCHAR) AS estrato,
    CAST(upm AS VARCHAR) AS upm,
    COALESCE(ponderador, 1.0) AS ponderador,

    -- VARIABLE OBJETIVO 1: Percepcion_Discriminacion (1 si respondió '1' o 'Sí' en s09a_01a .. s09a_01l)
    CASE WHEN (
        LOWER(TRIM(CAST(s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01b AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01c AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01d AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01e AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01f AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01g AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01h AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01i AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01j AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01k AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR
        LOWER(TRIM(CAST(s09a_01l AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)'
    ) THEN 1 ELSE 0 END AS Percepcion_Discriminacion,

    -- Motivos específicos individuales para análisis de factores
    CASE WHEN LOWER(TRIM(CAST(s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_a,
    CASE WHEN LOWER(TRIM(CAST(s09a_01b AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_b,
    CASE WHEN LOWER(TRIM(CAST(s09a_01c AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_c,
    CASE WHEN LOWER(TRIM(CAST(s09a_01d AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_d,
    CASE WHEN LOWER(TRIM(CAST(s09a_01e AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_e,
    CASE WHEN LOWER(TRIM(CAST(s09a_01f AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS disc_motivo_f,

    -- VARIABLE OBJETIVO 2: Victimizacion (1 si sufrió delito en s09b_02a o s09b_02b)
    CASE WHEN (
        (s09b_02a IS NOT NULL AND LOWER(TRIM(CAST(s09b_02a AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', '')) OR
        (s09b_02b IS NOT NULL AND LOWER(TRIM(CAST(s09b_02b AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', ''))
    ) THEN 1 ELSE 0 END AS Victimizacion,

    -- Percepción de Inseguridad (s09b_01: '3', '4' o contiene 'insegur')
    CASE WHEN (
        LOWER(CAST(s09b_01 AS VARCHAR)) LIKE '%insegur%' OR 
        TRIM(CAST(s09b_01 AS VARCHAR)) IN ('3', '4')
    ) THEN 1 ELSE 0 END AS Percepcion_Inseguridad,

    -- Denuncia del delito (s09b_03)
    CASE WHEN LOWER(TRIM(CAST(s09b_03 AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS Denuncio_Delito,

    CAST(s09a_04 AS VARCHAR) AS conoce_ley_045,
    CAST(s09b_01 AS VARCHAR) AS nivel_seguridad_cat,
    CAST(s09b_02a AS VARCHAR) AS delito_principal,
    CAST(s09b_04 AS VARCHAR) AS motivo_no_denuncia

FROM raw
WHERE ponderador IS NOT NULL;
"""
con.execute(query_etl)

# ==============================================================================
# 3. RESPUESTA DESCRIPTIVA PONDERADA (PREGUNTA DE INVESTIGACIÓN)
# ==============================================================================
print("\n=== 1. INDICADORES DEPARTAMENTALES PONDERADOS (BOLIVIA) ===")
df_depto = con.execute("""
    SELECT 
        departamento,
        COUNT(*) AS n_muestra,
        ROUND(SUM(ponderador), 0) AS poblacion_proyectada,
        ROUND(SUM(Percepcion_Discriminacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_discriminacion,
        ROUND(SUM(Victimizacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_victimizacion,
        ROUND(SUM(Percepcion_Inseguridad * ponderador) / SUM(ponderador) * 100, 2) AS pct_inseguridad,
        ROUND(SUM(CASE WHEN Victimizacion = 1 AND Denuncio_Delito = 0 THEN ponderador ELSE 0 END) / 
              NULLIF(SUM(Victimizacion * ponderador), 0) * 100, 2) AS pct_cifra_negra
    FROM base_analitica
    GROUP BY departamento
    ORDER BY pct_discriminacion DESC
""").df()
print(df_depto.to_string(index=False))

print("\n=== 2. BRECHA GEOGRÁFICA URBANA VS. RURAL ===")
df_area = con.execute("""
    SELECT 
        area_geo,
        ROUND(SUM(Percepcion_Discriminacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_discriminacion,
        ROUND(SUM(Victimizacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_victimizacion,
        ROUND(SUM(Percepcion_Inseguridad * ponderador) / SUM(ponderador) * 100, 2) AS pct_inseguridad
    FROM base_analitica
    GROUP BY area_geo
""").df()
print(df_area.to_string(index=False))

# ==============================================================================
# 4. ENTREGABLE 1: MODELO PREDICTIVO DE DISCRIMINACIÓN (LOGIT + RANDOM FOREST)
# ==============================================================================
df_ml = con.execute("""
    SELECT 
        Percepcion_Discriminacion,
        Victimizacion,
        Percepcion_Inseguridad,
        departamento,
        area_geo,
        estrato,
        ponderador
    FROM base_analitica
""").df()

print("\n=== 3. REGRESIÓN LOGÍSTICA PONDERADA: FACTORES ASOCIADOS A DISCRIMINACIÓN ===")
logit_disc = smf.glm(
    formula="Percepcion_Discriminacion ~ C(departamento) + C(area_geo) + Victimizacion + Percepcion_Inseguridad",
    data=df_ml,
    family=sm.families.Binomial(),
    freq_weights=df_ml['ponderador']
).fit()

res_disc = pd.DataFrame({
    'Odds_Ratio': np.exp(logit_disc.params),
    'p_valor': logit_disc.pvalues
}).round(4)
print(res_disc)

# Random Forest Clasificador
X = pd.get_dummies(df_ml[['departamento', 'area_geo', 'Victimizacion', 'Percepcion_Inseguridad', 'estrato']], drop_first=True)
y = df_ml['Percepcion_Discriminacion']
w = df_ml['ponderador']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.25, random_state=42, stratify=y
)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train, sample_weight=w_train)

y_prob = rf_model.predict_proba(X_test)[:, 1]
print(f"\n>>> Capacidad Predictiva Random Forest (AUC-ROC ponderado): {roc_auc_score(y_test, y_prob, sample_weight=w_test):.4f}")

# Top 10 variables más importantes en el modelo predictivo
importancias = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
print("\nTop 10 Predictores más importantes de Discriminación:")
print(importancias.round(4))

# ==============================================================================
# 5. ENTREGABLE 2: ANÁLISIS DE FACTORES DE RIESGO DELICTIVO (VICTIMIZACIÓN)
# ==============================================================================
print("\n=== 4. ANÁLISIS DE RIESGO DELICTIVO (ODDS RATIOS DE VICTIMIZACIÓN) ===")
logit_delito = smf.glm(
    formula="Victimizacion ~ C(departamento) + C(area_geo) + Percepcion_Discriminacion + Percepcion_Inseguridad",
    data=df_ml,
    family=sm.families.Binomial(),
    freq_weights=df_ml['ponderador']
).fit()
print(pd.DataFrame({'Odds_Ratio': np.exp(logit_delito.params), 'p_valor': logit_delito.pvalues}).round(4))

print("\n=== TOP DELITOS REPORTADOS (s09b_02a) Y MOTIVOS DE NO DENUNCIA (s09b_04) ===")
print(con.execute("""
    SELECT delito_principal, ROUND(SUM(ponderador), 0) AS casos_estimados
    FROM base_analitica
    WHERE Victimizacion = 1
    GROUP BY delito_principal
    ORDER BY casos_estimados DESC
    LIMIT 5
""").df().to_string(index=False))

# ==============================================================================
# 6. ENTREGABLE 3: CLUSTERING REGIONAL POR SEGURIDAD Y DISCRIMINACIÓN
# ==============================================================================
cols_cluster = ['pct_discriminacion', 'pct_victimizacion', 'pct_inseguridad', 'pct_cifra_negra']
X_clust = StandardScaler().fit_transform(df_depto[cols_cluster].fillna(0))

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_depto['cluster_id'] = kmeans.fit_predict(X_clust)

# Ordenar clusters de menor a mayor victimización para etiquetarlos con sentido analítico
orden = df_depto.groupby('cluster_id')['pct_victimizacion'].mean().sort_values().index
etiquetas_cluster = {
    orden[0]: 'Cluster 1: Riesgo Bajo / Mayor Cohesión',
    orden[1]: 'Cluster 2: Riesgo Moderado / Tensión Social',
    orden[2]: 'Cluster 3: Alta Vulnerabilidad Delictiva e Inseguridad'
}
df_depto['Perfil_Regional'] = df_depto['cluster_id'].map(etiquetas_cluster)

print("\n=== 5. CLUSTERING REGIONAL DE SEGURIDAD CIUDADANA ===")
print(df_depto[['departamento', 'Perfil_Regional', 'pct_discriminacion', 'pct_victimizacion', 'pct_inseguridad', 'pct_cifra_negra']].to_string(index=False))

# ==============================================================================
# 7. ENTREGABLE 4: MAPA INTERACTIVO DE PERCEPCIÓN DE INSEGURIDAD EN BOLIVIA
# ==============================================================================
coords_bolivia = pd.DataFrame({
    'departamento': ['La Paz', 'Santa Cruz', 'Cochabamba', 'Potosí', 'Chuquisaca', 'Oruro', 'Tarija', 'Beni', 'Pando'],
    'lat': [-16.5000, -17.7833, -17.3895, -19.5836, -19.0333, -17.9833, -21.5355, -14.8333, -11.0267],
    'lon': [-68.1500, -63.1821, -66.1568, -65.7531, -65.2627, -67.1167, -64.7296, -64.9000, -68.7692]
})

df_mapa = pd.merge(df_depto, coords_bolivia, on='departamento', how='inner')

fig = px.scatter_geo(
    df_mapa,
    lat='lat',
    lon='lon',
    color='pct_inseguridad',
    size='poblacion_proyectada',
    hover_name='departamento',
    hover_data={
        'Perfil_Regional': True,
        'pct_inseguridad': ':.2f',
        'pct_victimizacion': ':.2f',
        'pct_discriminacion': ':.2f',
        'pct_cifra_negra': ':.2f',
        'lat': False, 'lon': False
    },
    color_continuous_scale='YlOrRd',
    size_max=45,
    title='Bolivia (EH 2025): Percepción de Inseguridad, Victimización y Discriminación por Departamento'
)

fig.update_geos(
    visible=True,
    resolution=50,
    scope="south america",
    center=dict(lat=-16.5, lon=-64.5),
    projection_scale=4.6,
    showcountries=True,
    countrycolor="Black",
    showsubunits=True
)

ruta_mapa = os.path.join(os.getcwd(), "mapa_inseguridad_bolivia.html")
fig.write_html(ruta_mapa)
print(f"\n[OK] Mapa interactivo generado exitosamente en:\n{ruta_mapa}")

<>:66: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:66: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_16932\107873930.py:66: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  LOWER(TRIM(CAST(s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí|1\. si|1\. sí)' OR


--- MUESTRA DE ETIQUETAS EN EH2025_Discriminacion.sav ---
        depto    area s09a_01a         s09b_01  \
0  Chuquisaca  Urbano    2. No  1.Muy inseguro   
1  Chuquisaca  Urbano    2. No        3.Seguro   
2  Chuquisaca  Urbano    2. No      2.Inseguro   
3  Chuquisaca  Urbano    2. No      2.Inseguro   
4  Chuquisaca  Urbano    1. Si        3.Seguro   

                                            s09b_02a  
0                                          8.ninguno  
1                                          8.ninguno  
2                                          8.ninguno  
3  1.robo o hurto, asalto o atraco de algún objet...  
4                                          8.ninguno  

=== 1. INDICADORES DEPARTAMENTALES PONDERADOS (BOLIVIA) ===
departamento  n_muestra  poblacion_proyectada  pct_discriminacion  pct_victimizacion  pct_inseguridad  pct_cifra_negra
      La Paz       2970                3168.0               24.07              100.0            53.95            97.52
       Pando

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


                               Odds_Ratio  p_valor
Intercept                          0.3602   0.0000
C(departamento)[T.Chuquisaca]      1.1893   0.3327
C(departamento)[T.Cochabamba]      1.1658   0.3018
C(departamento)[T.La Paz]          2.3533   0.0000
C(departamento)[T.Oruro]           1.1591   0.4056
C(departamento)[T.Pando]           1.6841   0.0459
C(departamento)[T.Potosí]          1.1680   0.3547
C(departamento)[T.Santa Cruz]      0.8863   0.4089
C(departamento)[T.Tarija]          0.6772   0.0524
C(area_geo)[T.Urbana]              0.8579   0.0118
Victimizacion                      0.3602   0.0000
Percepcion_Inseguridad             1.2998   0.0000

>>> Capacidad Predictiva Random Forest (AUC-ROC ponderado): 0.6141

Top 10 Predictores más importantes de Discriminación:
departamento_La Paz        0.2890
Percepcion_Inseguridad     0.1042
estrato_221                0.0931
estrato_213                0.0696
departamento_Santa Cruz    0.0639
estrato_714                0.0350
departamen

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_ir

                                 Odds_Ratio  p_valor
Intercept                      6.924372e+12   0.9997
C(departamento)[T.Chuquisaca]  1.000000e+00   1.0000
C(departamento)[T.Cochabamba]  1.000000e+00   1.0000
C(departamento)[T.La Paz]      1.000000e+00   1.0000
C(departamento)[T.Oruro]       1.000000e+00   1.0000
C(departamento)[T.Pando]       1.000000e+00   1.0000
C(departamento)[T.Potosí]      1.000000e+00   1.0000
C(departamento)[T.Santa Cruz]  1.000000e+00   1.0000
C(departamento)[T.Tarija]      1.000000e+00   1.0000
C(area_geo)[T.Urbana]          1.000000e+00   1.0000
Percepcion_Discriminacion      1.000000e+00   1.0000
Percepcion_Inseguridad         1.000000e+00   1.0000

=== TOP DELITOS REPORTADOS (s09b_02a) Y MOTIVOS DE NO DENUNCIA (s09b_04) ===
                                                                                 delito_principal  casos_estimados
                                                                                        8.ninguno          10671.0
   

In [11]:
%pip install duckdb pandas pyreadstat dask[dataframe] ydata-profiling ipywidgets

Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 4.0.0 Requires-Python >=3.7,<3.11; 4.1.0 Requires-Python >=3.7,<3.12; 4.1.1 Requires-Python >=3.7,<3.12; 4.1.2 Requires-Python >=3.7,<3.12; 4.10.0 Requires-Python >=3.7,<3.13; 4.11.0 Requires-Python >=3.7,<3.13; 4.12.0 Requires-Python >=3.7,<3.13; 4.12.1 Requires-Python >=3.7,<3.13; 4.12.2 Requires-Python >=3.7,<3.13; 4.13.0 Requires-Python >=3.7,<3.13; 4.14.0 Requires-Python >=3.7,<3.13; 4.15.0 Requires-Python >=3.7,<3.13; 4.15.1 Requires-Python >=3.7,<3.13; 4.16.0 Requires-Python >=3.7,<3.13; 4.16.1 Requires-Python >=3.7,<3.13; 4.17.0 Requires-Python >=3.7,<3.14; 4.18.0 Requires-Python >=3.10,<3.14; 4.18.1 Requires-Python >=3.10,<3.14; 4.18.4 Requires-Python >=3.10,<3.14; 4.2.0 Requires-Python >=3.7,<3.12; 4.3.0 Requires-Python >=3.7,<3.12; 4.3.1 Requires-Python >=3.7,<3.12; 4.3.2 Requires-Python >=3.7,<3.12; 4.4.0 Requires-Python >=3.7,<3.12; 4.5.0 Requires-Python >=3.7,<3.12; 4.5.1 Requires-Python >=3.7,

In [16]:
%pip install --upgrade pip setuptools wheel
%pip install ydata-profiling duckdb pandas pyreadstat dask[dataframe] ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 4.0.0 Requires-Python >=3.7,<3.11; 4.1.0 Requires-Python >=3.7,<3.12; 4.1.1 Requires-Python >=3.7,<3.12; 4.1.2 Requires-Python >=3.7,<3.12; 4.10.0 Requires-Python >=3.7,<3.13; 4.11.0 Requires-Python >=3.7,<3.13; 4.12.0 Requires-Python >=3.7,<3.13; 4.12.1 Requires-Python >=3.7,<3.13; 4.12.2 Requires-Python >=3.7,<3.13; 4.13.0 Requires-Python >=3.7,<3.13; 4.14.0 Requires-Python >=3.7,<3.13; 4.15.0 Requires-Python >=3.7,<3.13; 4.15.1 Requires-Python >=3.7,<3.13; 4.16.0 Requires-Python >=3.7,<3.13; 4.16.1 Requires-Python >=3.7,<3.13; 4.17.0 Requires-Python >=3.7,<3.14; 4.18.0 Requires-Python >=3.10,<3.14; 4.18.1 Requires-Python >=3.10,<3.14; 4.18.4 Requires-Python >=3.10,<3.14; 4.2.0 Requires-Python >=3.7,<3.12; 4.3.0 Requires-Python >=3.7,<3.12; 4.3.1 Requires-Python >=3.7,<3.12; 4.3.2 Requires-Python >=3.7,<3.12; 4.4.0 Requires-Python >=3.7,<3.12; 4.5.0 Requires-Python >=3.7,<3.12; 4.5.1 Requires-Python >=3.7,

In [19]:
import sys
!{sys.executable} -m pip install dask distributed pyarrow
!{sys.executable} -m pip install sweetviz
!{sys.executable} -m pip install ydata-profiling

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 9.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 10.1 MB/s  0:00:00
   ---------------------------------------- 0.0/29.8 MB ? eta -:--:--
   -- ------------------------------------- 2.1/29.8 MB 11.1 MB/s eta 0:00:03
   ----- ---------------------------------- 4.5/29.8 MB 10.9 MB/s eta 0:00:03
   -------- ------------------------------- 6.6/29.8 MB 11.0 MB/s eta 0:00:03
   ------------ --------------------------- 9.2/29.8 MB 11.2 MB/s eta 0:00:02
   -------------- ------------------------- 11.0/29.8 MB 10.7 MB/s eta 0:00:02
   ------------------ --------------------- 13.6/29.8 MB 11.0 MB/s eta 0:00:02
   --------------------- ------------------ 16.0/29.8 MB 11.0 MB/s eta 0:00:02
   ------------------------ --------------- 18.1/29.8 MB 11.0 MB/s eta 0:00:02
   --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/15.1 MB 10.3 MB/s eta 0:00:02
   ----------- ---------------------------- 4.2/15.1 MB 10.2 MB/s eta 0:00:02
   ---------------- ----------------------- 6.3/15.1 MB 9.9 MB/s eta 0:00:01
   ---------------------- ----------------- 8.4/15.1 MB 10.1 MB/s eta 0:00:01
   --------------------------- ------------ 10.5/15.1 MB 10.1 MB/s eta 0:00:01
   --------------------------------- ------ 12.6/15.1 MB 10.1 MB/s eta 0:00:01
   -------------------------------------- - 14.7/15.1 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 15.1/15.1 MB 9.8 MB/s  0:00:01

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [importlib_resources]
   -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Ignored the following versions that require a different python version: 4.0.0 Requires-Python >=3.7,<3.11; 4.1.0 Requires-Python >=3.7,<3.12; 4.1.1 Requires-Python >=3.7,<3.12; 4.1.2 Requires-Python >=3.7,<3.12; 4.10.0 Requires-Python >=3.7,<3.13; 4.11.0 Requires-Python >=3.7,<3.13; 4.12.0 Requires-Python >=3.7,<3.13; 4.12.1 Requires-Python >=3.7,<3.13; 4.12.2 Requires-Python >=3.7,<3.13; 4.13.0 Requires-Python >=3.7,<3.13; 4.14.0 Requires-Python >=3.7,<3.13; 4.15.0 Requires-Python >=3.7,<3.13; 4.15.1 Requires-Python >=3.7,<3.13; 4.16.0 Requires-Python >=3.7,<3.13; 4.16.1 Requires-Python >=3.7,<3.13; 4.17.0 Requires-Python >=3.7,<3.14; 4.18.0 Requires-Python >=3.10,<3.14; 4.18.1 Requires-Python >=3.10,<3.14; 4.18.4 Requires-Python >=3.10,<3.14; 4.2.0 Requires-Python >=3.7,<3.12; 4.3.0 Requires-Python >=3.7,<3.12; 4.3.1 Requires-Python >=3.7,<3.12; 4.3.2 Requires-Py

In [3]:
import os
import glob
import gc
import duckdb
import pandas as pd
import pyreadstat
import dask.dataframe as dd
from dask import delayed

# Detección automática entre YData-Profiling y Sweetviz según compatibilidad del entorno
try:
    from ydata_profiling import ProfileReport
    MOTOR_EDA = "ydata"
except ImportError:
    import sweetviz as sv
    MOTOR_EDA = "sweetviz"

print(f"Motor EDA activo: {MOTOR_EDA}")

# ==============================================================================
# 1. CONFIGURACIÓN DE DIRECTORIOS Y LÍMITES DE MEMORIA EN DUCKDB
# ==============================================================================
BASE_DIR = r"C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025"
os.chdir(BASE_DIR)
os.makedirs("outputs", exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute("PRAGMA memory_limit='2GB';")
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA temp_directory='outputs/duckdb_temp';")

# ==============================================================================
# 2. LECTURA OPTIMIZADA DE ARCHIVOS .SAV (SPSS) CON PYREADSTAT + DASK
# ==============================================================================
archivos_sav = glob.glob("*.sav")
print("Archivos .sav detectados:", archivos_sav)

def leer_sav_optimizado(ruta_sav, columnas_necesarias=None):
    """Lee un .sav de SPSS aplicando etiquetas de valor y proyección temprana de columnas."""
    df_part, _ = pyreadstat.read_sav(
        ruta_sav,
        usecols=columnas_necesarias,
        apply_value_formats=True
    )
    for col in df_part.select_dtypes(include=["category", "object"]).columns:
        df_part[col] = df_part[col].astype(str)
    return df_part

ruta_disc = "EH2025_Discriminacion.sav"
ruta_pers = "EH2025_Personas.sav"

# Carga mediante Dask delayed para gestión controlada de memoria
ddf_disc = dd.from_delayed([delayed(leer_sav_optimizado)(ruta_disc)])
df_disc_mem = ddf_disc.compute()
con.register("tbl_discriminacion", df_disc_mem)

# Cruce con EH2025_Personas.sav si existe en la carpeta
tiene_personas = os.path.exists(ruta_pers)
if tiene_personas:
    _, meta_pers = pyreadstat.read_sav(ruta_pers, metadataonly=True)
    cols_posibles = ["folio", "nro", "s01a_02", "s01a_03", "s01a_09", "niv_ed", "p0", "sexo", "edad"]
    cols_validas = [c for c in cols_posibles if c in meta_pers.column_names]
    
    ddf_pers = dd.from_delayed([delayed(leer_sav_optimizado)(ruta_pers, cols_validas)])
    df_pers_mem = ddf_pers.compute()
    con.register("tbl_personas", df_pers_mem)
    print(f"Módulo Personas cargado ({len(cols_validas)} columnas): {cols_validas}")

# ==============================================================================
# 3. CONSULTA DE CRUCE (JOIN) EN DUCKDB Y EXPORTACIÓN A PARQUET COMPRIMIDO
# ==============================================================================
if tiene_personas:
    query_join = """
    COPY (
        SELECT 
            d.folio,
            d.nro,
            d.depto,
            d.area,
            d.estrato,
            d.upm,
            d.ponderador,
            p.* EXCLUDE (folio, nro),
            CASE WHEN (
                LOWER(TRIM(CAST(d.s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01b AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01c AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01d AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01e AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01f AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01g AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01h AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01i AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01j AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01k AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(d.s09a_01l AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)'
            ) THEN 1 ELSE 0 END AS Percepcion_Discriminacion,
            CASE WHEN (
                (d.s09b_02a IS NOT NULL AND LOWER(TRIM(CAST(d.s09b_02a AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', '')) OR
                (d.s09b_02b IS NOT NULL AND LOWER(TRIM(CAST(d.s09b_02b AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', ''))
            ) THEN 1 ELSE 0 END AS Victimizacion,
            d.s09a_02, d.s09a_03_1, d.s09a_04, d.s09b_01, d.s09b_02a, d.s09b_03, d.s09b_04
        FROM tbl_discriminacion d
        LEFT JOIN tbl_personas p
            ON TRIM(CAST(d.folio AS VARCHAR)) = TRIM(CAST(p.folio AS VARCHAR))
           AND CAST(d.nro AS INTEGER) = CAST(p.nro AS INTEGER)
        WHERE d.ponderador IS NOT NULL
    ) TO 'outputs/base_integrada_eh2025.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD');
    """
else:
    query_join = """
    COPY (
        SELECT 
            folio, nro, depto, area, estrato, upm, ponderador,
            CASE WHEN (
                LOWER(TRIM(CAST(s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01b AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01c AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01d AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01e AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01f AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01g AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01h AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01i AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01j AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01k AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
                LOWER(TRIM(CAST(s09a_01l AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)'
            ) THEN 1 ELSE 0 END AS Percepcion_Discriminacion,
            CASE WHEN (
                (s09b_02a IS NOT NULL AND LOWER(TRIM(CAST(s09b_02a AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', '')) OR
                (s09b_02b IS NOT NULL AND LOWER(TRIM(CAST(s09b_02b AS VARCHAR))) NOT IN ('0', '2', 'no', '2. no', 'ninguno', 'nan', 'none', ''))
            ) THEN 1 ELSE 0 END AS Victimizacion,
            s09a_02, s09a_03_1, s09a_04, s09b_01, s09b_02a, s09b_03, s09b_04
        FROM tbl_discriminacion
        WHERE ponderador IS NOT NULL
    ) TO 'outputs/base_integrada_eh2025.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD');
    """

con.execute(query_join)

# Liberación explícita de vistas registradas y memoria RAM
con.unregister("tbl_discriminacion")
del df_disc_mem
if tiene_personas:
    con.unregister("tbl_personas")
    del df_pers_mem
gc.collect()

# ==============================================================================
# 4. EDA AUTOMATIZADO INTERACTIVO EN outputs/
# ==============================================================================
df_eda = con.execute("SELECT * FROM 'outputs/base_integrada_eh2025.parquet'").df()
ruta_reporte = os.path.join("outputs", "eda_discriminacion_seguridad_eh2025.html")

if MOTOR_EDA == "ydata":
    profile = ProfileReport(
        df_eda,
        title="EDA Caso 3: Discriminación y Seguridad Ciudadana - EH 2025 Bolivia (.sav)",
        explorative=True,
        correlations={
            "pearson": {"calculate": True},
            "spearman": {"calculate": True},
            "kendall": {"calculate": False},
            "phi_k": {"calculate": False},
            "cramers": {"calculate": True}
        }
    )
    profile.to_file(ruta_reporte)
else:
    reporte_sv = sv.analyze(df_eda, target_feat="Percepcion_Discriminacion")
    reporte_sv.show_html(filepath=ruta_reporte, open_browser=False)

print(f"[OK] Base Parquet guardada en: {os.path.abspath('outputs/base_integrada_eh2025.parquet')}")
print(f"[OK] Reporte EDA interactivo guardado en: {os.path.abspath(ruta_reporte)}")

<>:85: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:116: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:85: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:116: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_10844\4177272176.py:85: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  LOWER(TRIM(CAST(d.s09a_01a AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' OR
C:\Users\user\AppData\Local\Temp\ipykernel_10844\4177272176.py:116: SyntaxWarning: "\." is 

Motor EDA activo: sweetviz
Archivos .sav detectados: ['EH2025_Discriminacion.sav', 'EH2025_Equipamiento.sav', 'EH2025_Gastos_Alimentarios.sav', 'EH2025_Gastos_No_Alimentarios.sav', 'EH2025_Persona.sav', 'EH2025_Seguridad_Alimentaria.sav', 'EH2025_Vivienda_1.sav']


C:\Users\user\AppData\Local\Temp\ipykernel_10844\4177272176.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_part.select_dtypes(include=["category", "object"]).columns:
C:\Users\user\AppData\Local\Temp\ipykernel_10844\4177272176.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.o

Report outputs\eda_discriminacion_seguridad_eh2025.html was generated.
[OK] Base Parquet guardada en: C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025\outputs\base_integrada_eh2025.parquet
[OK] Reporte EDA interactivo guardado en: C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025\outputs\eda_discriminacion_seguridad_eh2025.html


In [4]:
import sys
!{sys.executable} -m pip install folium plotly scikit-learn duckdb pandas numpy nbformat


   ----------------------------------------  0/15 [xyzservices]
   -- -------------------------------------  1/15 [urllib3]
   -- -------------------------------------  1/15 [urllib3]
   -- -------------------------------------  1/15 [urllib3]
   -- -------------------------------------  1/15 [urllib3]
   -- -------------------------------------  1/15 [urllib3]
   -------- -------------------------------  3/15 [idna]
   -------- -------------------------------  3/15 [idna]
   -------- -------------------------------  3/15 [idna]
   ---------- -----------------------------  4/15 [fastjsonschema]
   ---------- -----------------------------  4/15 [fastjsonschema]
   ------------- --------------------------  5/15 [charset_normalizer]
   ------------- --------------------------  5/15 [charset_normalizer]
   ------------- --------------------------  5/15 [charset_normalizer]
   ------------- --------------------------  5/15 [charset_normalizer]
   ---------------- -----------------------  6

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
import os
import duckdb
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ==============================================================================
# 0. CONEXIÓN A LA BASE PARQUET INTEGRADA EN DUCKDB
# ==============================================================================
BASE_DIR = r"C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025"
os.chdir(BASE_DIR)
os.makedirs("outputs", exist_ok=True)

con = duckdb.connect(database=":memory:")
RUTA_PARQUET = "outputs/base_integrada_eh2025.parquet"

# Normalizamos dimensiones clave en una vista analítica limpia dentro de DuckDB
con.execute(f"""
CREATE OR REPLACE VIEW v_analisis AS
SELECT
    folio,
    nro,
    CASE 
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%chuquisaca%' OR TRIM(CAST(depto AS VARCHAR)) = '1' THEN 'Chuquisaca'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%la paz%'     OR TRIM(CAST(depto AS VARCHAR)) = '2' THEN 'La Paz'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%cochabamba%' OR TRIM(CAST(depto AS VARCHAR)) = '3' THEN 'Cochabamba'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%oruro%'      OR TRIM(CAST(depto AS VARCHAR)) = '4' THEN 'Oruro'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%potos%'      OR TRIM(CAST(depto AS VARCHAR)) = '5' THEN 'Potosí'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%tarija%'     OR TRIM(CAST(depto AS VARCHAR)) = '6' THEN 'Tarija'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%santa cruz%' OR TRIM(CAST(depto AS VARCHAR)) = '7' THEN 'Santa Cruz'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%beni%'       OR TRIM(CAST(depto AS VARCHAR)) = '8' THEN 'Beni'
        WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%pando%'      OR TRIM(CAST(depto AS VARCHAR)) = '9' THEN 'Pando'
        ELSE CAST(depto AS VARCHAR)
    END AS departamento,
    CASE 
        WHEN LOWER(CAST(area AS VARCHAR)) LIKE '%urban%' OR TRIM(CAST(area AS VARCHAR)) = '1' THEN 'Urbana'
        ELSE 'Rural' 
    END AS area_geo,
    CASE WHEN (
        LOWER(CAST(s09b_01 AS VARCHAR)) LIKE '%insegur%' OR TRIM(CAST(s09b_01 AS VARCHAR)) IN ('3', '4')
    ) THEN 1 ELSE 0 END AS Percepcion_Inseguridad,
    CASE WHEN LOWER(TRIM(CAST(s09b_03 AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS Denuncio_Delito,
    COALESCE(NULLIF(TRIM(CAST(s09b_01 AS VARCHAR)), 'nan'), 'Sin dato') AS nivel_seguridad_lbl,
    Percepcion_Discriminacion,
    Victimizacion,
    COALESCE(ponderador, 1.0) AS ponderador
FROM '{RUTA_PARQUET}';
""")

# ==============================================================================
# VISUALIZACIÓN 1: MAPA GEOESPACIAL INTERACTIVO CON CAPAS COMPLEMENTARIAS (FOLIUM)
# ==============================================================================
df_geo = con.execute("""
    SELECT 
        departamento,
        ROUND(SUM(ponderador), 0) AS poblacion_expandida,
        ROUND(SUM(Percepcion_Discriminacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_discriminacion,
        ROUND(SUM(Victimizacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_victimizacion,
        ROUND(SUM(Percepcion_Inseguridad * ponderador) / SUM(ponderador) * 100, 2) AS pct_inseguridad,
        ROUND(SUM(CASE WHEN Victimizacion = 1 AND Denuncio_Delito = 0 THEN ponderador ELSE 0 END) / 
              NULLIF(SUM(Victimizacion * ponderador), 0) * 100, 2) AS pct_cifra_negra
    FROM v_analisis
    GROUP BY departamento
""").df()

coords = pd.DataFrame({
    'departamento': ['La Paz', 'Santa Cruz', 'Cochabamba', 'Potosí', 'Chuquisaca', 'Oruro', 'Tarija', 'Beni', 'Pando'],
    'lat': [-16.5000, -17.7833, -17.3895, -19.5836, -19.0333, -17.9833, -21.5355, -14.8333, -11.0267],
    'lon': [-68.1500, -63.1821, -66.1568, -65.7531, -65.2627, -67.1167, -64.7296, -64.9000, -68.7692]
})
df_geo = pd.merge(df_geo, coords, on='departamento', how='inner')

mapa_bolivia = folium.Map(location=[-16.5, -64.5], zoom_start=6, tiles="CartoDB positron")

# Capa 1: Tasa de Percepción de Discriminación (Variable Objetivo 1)
capa_disc = folium.FeatureGroup(name="Capa 1: Percepción de Discriminación (%)", show=True)
# Capa 2: Tasa de Victimización Delictiva (Variable Objetivo 2)
capa_vict = folium.FeatureGroup(name="Capa 2: Victimización Delictiva (%)", show=False)
# Capa 3: Percepción de Inseguridad y Cifra Negra
capa_inseg = folium.FeatureGroup(name="Capa 3: Percepción de Inseguridad (%)", show=False)

for _, row in df_geo.iterrows():
    popup_html = f"""
    <div style="font-family: Arial; width: 230px;">
        <h4 style="margin-bottom:5px; color:#1f4e78;">{row['departamento']}</h4>
        <b>Población estimada:</b> {int(row['poblacion_expandida']):,}<br>
        <hr style="margin:4px 0;">
        <b>Discriminación:</b> {row['pct_discriminacion']}%<br>
        <b>Victimización:</b> {row['pct_victimizacion']}%<br>
        <b>Inseguridad percibida:</b> {row['pct_inseguridad']}%<br>
        <b>Cifra negra (no denuncia):</b> {row['pct_cifra_negra']}%
    </div>
    """
    # Marcador Capa 1 (Discriminación)
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(10, row['pct_discriminacion'] * 1.4),
        color="#d95f02", fill=True, fill_color="#d95f02", fill_opacity=0.65,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row['departamento']} | Discriminación: {row['pct_discriminacion']}%"
    ).add_to(capa_disc)

    # Marcador Capa 2 (Victimización)
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(10, row['pct_victimizacion'] * 1.6),
        color="#7570b3", fill=True, fill_color="#7570b3", fill_opacity=0.65,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row['departamento']} | Victimización: {row['pct_victimizacion']}%"
    ).add_to(capa_vict)

    # Marcador Capa 3 (Inseguridad)
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(10, row['pct_inseguridad'] * 0.7),
        color="#e7298a", fill=True, fill_color="#e7298a", fill_opacity=0.55,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{row['departamento']} | Inseguridad: {row['pct_inseguridad']}%"
    ).add_to(capa_inseg)

capa_disc.add_to(mapa_bolivia)
capa_vict.add_to(mapa_bolivia)
capa_inseg.add_to(mapa_bolivia)
folium.LayerControl(collapsed=False).add_to(mapa_bolivia)

mapa_bolivia.save("outputs/viz1_mapa_geoespacial_bolivia.html")
print("[OK] Viz 1 guardada: outputs/viz1_mapa_geoespacial_bolivia.html")

# ==============================================================================
# VISUALIZACIÓN 2: SUNBURST JERÁRQUICO MULTIVARIADO (PLOTLY)
# ==============================================================================
df_sunburst = con.execute("""
    SELECT 
        departamento,
        area_geo,
        CASE WHEN Percepcion_Discriminacion = 1 THEN 'Con Discriminación' ELSE 'Sin Discriminación' END AS estado_discriminacion,
        ROUND(SUM(ponderador), 0) AS poblacion,
        ROUND(SUM(Victimizacion * ponderador) / SUM(ponderador) * 100, 2) AS tasa_victimizacion
    FROM v_analisis
    GROUP BY departamento, area_geo, estado_discriminacion
""").df()

fig_sunburst = px.sunburst(
    df_sunburst,
    path=['departamento', 'area_geo', 'estado_discriminacion'],
    values='poblacion',
    color='tasa_victimizacion',
    color_continuous_scale='RdYlBu_r',
    title='Distribución Jerárquica: Departamento → Área → Discriminación (Color: % Victimización)',
    hover_data={'tasa_victimizacion': ':.2f'}
)
fig_sunburst.update_layout(margin=dict(t=50, l=10, r=10, b=10))
fig_sunburst.write_html("outputs/viz2_sunburst_multivariado.html")
fig_sunburst.show()

# ==============================================================================
# VISUALIZACIÓN 3: DISPERSIÓN MULTIVARIADA Y BARRAS AGRUPADAS (PLOTLY)
# ==============================================================================
df_scatter = con.execute("""
    SELECT 
        departamento,
        area_geo,
        ROUND(SUM(ponderador), 0) AS poblacion_expandida,
        ROUND(SUM(Percepcion_Discriminacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_discriminacion,
        ROUND(SUM(Victimizacion * ponderador) / SUM(ponderador) * 100, 2) AS pct_victimizacion,
        ROUND(SUM(Percepcion_Inseguridad * ponderador) / SUM(ponderador) * 100, 2) AS pct_inseguridad
    FROM v_analisis
    GROUP BY departamento, area_geo
""").df()

fig_scatter = px.scatter(
    df_scatter,
    x='pct_discriminacion',
    y='pct_victimizacion',
    size='poblacion_expandida',
    color='area_geo',
    text='departamento',
    facet_col='area_geo',
    hover_data=['pct_inseguridad', 'poblacion_expandida'],
    labels={
        'pct_discriminacion': 'Percepción de Discriminación (%)',
        'pct_victimizacion': 'Victimización Delictiva (%)',
        'area_geo': 'Área Geográfica'
    },
    title='Relación Multivariada: Discriminación vs. Victimización por Departamento y Área (Tamaño: Población)',
    size_max=45
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.write_html("outputs/viz3_dispersion_multivariada.html")
fig_scatter.show()

# ==============================================================================
# VISUALIZACIÓN 4: SEGMENTACIÓN DE INDIVIDUOS/HOGARES POR CLUSTERING (K-MEANS + PCA)
# ==============================================================================
# Extraemos matriz de características a nivel individual/hogar desde DuckDB
df_ind = con.execute("""
    SELECT 
        folio,
        nro,
        departamento,
        area_geo,
        Percepcion_Discriminacion,
        Victimizacion,
        Percepcion_Inseguridad,
        Denuncio_Delito,
        ponderador
    FROM v_analisis
""").df()

features_ind = ['Percepcion_Discriminacion', 'Victimizacion', 'Percepcion_Inseguridad', 'Denuncio_Delito']
X_ind = StandardScaler().fit_transform(df_ind[features_ind])

# Segmentación K-Means en 4 perfiles socio-delictivos
kmeans_ind = KMeans(n_clusters=4, random_state=42, n_init=10)
df_ind['cluster_id'] = kmeans_ind.fit_predict(X_ind)

# Interpretación automática de los perfiles obtenidos (centroides ponderados)
perfil_tabla = df_ind.groupby('cluster_id').agg(
    n_muestra=('folio', 'count'),
    pct_disc=('Percepcion_Discriminacion', lambda s: round(s.mean() * 100, 1)),
    pct_vict=('Victimizacion', lambda s: round(s.mean() * 100, 1)),
    pct_inseg=('Percepcion_Inseguridad', lambda s: round(s.mean() * 100, 1)),
    pct_denuncia=('Denuncio_Delito', lambda s: round(s.mean() * 100, 1))
).reset_index()

print("\n=== PERFILES DE SEGMENTACIÓN DE INDIVIDUOS/HOGARES (K-MEANS) ===")
print(perfil_tabla.to_string(index=False))

# Asignamos nombres descriptivos según el patrón dominante de cada clúster
nombres_perfil = {}
for _, r in perfil_tabla.iterrows():
    cid = int(r['cluster_id'])
    if r['pct_vict'] > 50 and r['pct_disc'] > 40:
        nombres_perfil[cid] = f"Perfil {cid}: Doble Vulnerabilidad (Víctima + Discriminado)"
    elif r['pct_vict'] > 50:
        nombres_perfil[cid] = f"Perfil {cid}: Víctima Delictiva Directa"
    elif r['pct_disc'] > 50:
        nombres_perfil[cid] = f"Perfil {cid}: Discriminación sin Victimización Delictiva"
    elif r['pct_inseg'] > 50:
        nombres_perfil[cid] = f"Perfil {cid}: Alta Inseguridad Percibida (No Víctima)"
    else:
        nombres_perfil[cid] = f"Perfil {cid}: Baja Vulnerabilidad / Cohesión Estable"

df_ind['Perfil_Segmento'] = df_ind['cluster_id'].map(nombres_perfil)

# Proyección visual 2D mediante PCA sobre una muestra estratificada de 3,000 registros para fluidez interactiva
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(X_ind)
df_ind['PCA_1'] = coords_pca[:, 0] + np.random.normal(0, 0.08, size=len(df_ind))  # Jitter leve para evitar solapamiento binario
df_ind['PCA_2'] = coords_pca[:, 1] + np.random.normal(0, 0.08, size=len(df_ind))

df_plot_pca = df_ind.sample(n=min(3000, len(df_ind)), random_state=42)

fig_cluster = px.scatter(
    df_plot_pca,
    x='PCA_1',
    y='PCA_2',
    color='Perfil_Segmento',
    symbol='area_geo',
    hover_data=['departamento', 'Percepcion_Discriminacion', 'Victimizacion', 'Percepcion_Inseguridad'],
    title='Segmentación de Individuos/Hogares por Vulnerabilidad (Proyección PCA de Clústeres K-Means)',
    opacity=0.75
)
fig_cluster.write_html("outputs/viz4_clustering_individuos_pca.html")
fig_cluster.show()
print("[OK] Todas las visualizaciones interactivas fueron guardadas en la carpeta outputs/.")

<>:78: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:78: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_10844\2939262420.py:78: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


[OK] Viz 1 guardada: outputs/viz1_mapa_geoespacial_bolivia.html



=== PERFILES DE SEGMENTACIÓN DE INDIVIDUOS/HOGARES (K-MEANS) ===
 cluster_id  n_muestra  pct_disc  pct_vict  pct_inseg  pct_denuncia
          0       5375       0.0     100.0        0.0           0.0
          1       4941       0.0     100.0      100.0           0.0
          2       1747     100.0     100.0       53.5           0.0
          3        295      26.4     100.0       69.2         100.0


[OK] Todas las visualizaciones interactivas fueron guardadas en la carpeta outputs/.


In [6]:
import sys
!{sys.executable} -m pip install lightgbm xgboost shap scikit-learn plotly duckdb pandas numpy

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 8.6 MB/s  0:00:00
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.8/48.9 MB 10.1 MB/s eta 0:00:05
   --- ------------------------------------ 3.9/48.9 MB 9.5 MB/s eta 0:00:05
   ---- ----------------------------------- 5.8/48.9 MB 9.3 MB/s eta 0:00:05
   ------ --------------------------------- 8.1/48.9 MB 9.5 MB/s eta 0:00:05
   -------- ------------------------------- 10.5/48.9 MB 9.8 MB/s eta 0:00:04
   ---------- ----------------------------- 12.6/48.9 MB 9.8 MB/s eta 0:00:04
   ----------- ---------------------------- 14.4/48.9 MB 9.6 MB/s eta 0:00:04
   ------------- -------------------------- 16.5/48.9 MB 9.6 MB/s eta 0:00:04
   -------------- ------------------------- 18.4/48.9 MB 9.5 MB/s eta 0:00:04
   ---------------- ----------------------- 20.2/48.9 MB 9.4 MB/s eta 0:00:04
   --------

In [7]:
import os
import re
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)
import lightgbm as lgb
import shap

# ==============================================================================
# 1. EXTRACCIÓN Y PREPARACIÓN DE FEATURES EN DUCKDB (SIN DATA LEAKAGE)
# ==============================================================================
BASE_DIR = r"C:\Users\user\Desktop\maestria\visualizacion\tarea2\BD_EH2025"
os.chdir(BASE_DIR)
os.makedirs("outputs", exist_ok=True)

con = duckdb.connect(database=":memory:")
RUTA_PARQUET = "outputs/base_integrada_eh2025.parquet"

# Seleccionamos predictores sociodemográficos, geográficos y de contexto de seguridad
# Nota: Excluimos s09a_01* (que definen el target) y s09a_02/s09a_03* (preguntas filtro solo respondidas por quienes ya dijeron Sí)
df_model = con.execute(f"""
    SELECT 
        Percepcion_Discriminacion AS target,
        COALESCE(ponderador, 1.0) AS ponderador,
        CASE 
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%chuquisaca%' OR TRIM(CAST(depto AS VARCHAR)) = '1' THEN 'Chuquisaca'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%la paz%'     OR TRIM(CAST(depto AS VARCHAR)) = '2' THEN 'La Paz'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%cochabamba%' OR TRIM(CAST(depto AS VARCHAR)) = '3' THEN 'Cochabamba'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%oruro%'      OR TRIM(CAST(depto AS VARCHAR)) = '4' THEN 'Oruro'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%potos%'      OR TRIM(CAST(depto AS VARCHAR)) = '5' THEN 'Potosí'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%tarija%'     OR TRIM(CAST(depto AS VARCHAR)) = '6' THEN 'Tarija'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%santa cruz%' OR TRIM(CAST(depto AS VARCHAR)) = '7' THEN 'Santa Cruz'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%beni%'       OR TRIM(CAST(depto AS VARCHAR)) = '8' THEN 'Beni'
            WHEN LOWER(CAST(depto AS VARCHAR)) LIKE '%pando%'      OR TRIM(CAST(depto AS VARCHAR)) = '9' THEN 'Pando'
            ELSE 'Otro'
        END AS departamento,
        CASE 
            WHEN LOWER(CAST(area AS VARCHAR)) LIKE '%urban%' OR TRIM(CAST(area AS VARCHAR)) = '1' THEN 'Urbana'
            ELSE 'Rural' 
        END AS area_geo,
        Victimizacion AS fue_victima_delito,
        CASE WHEN (
            LOWER(CAST(s09b_01 AS VARCHAR)) LIKE '%insegur%' OR TRIM(CAST(s09b_01 AS VARCHAR)) IN ('3', '4')
        ) THEN 1 ELSE 0 END AS percibe_inseguridad,
        CASE WHEN LOWER(TRIM(CAST(s09a_04 AS VARCHAR))) SIMILAR TO '(1|1\..*|si|sí.*)' THEN 1 ELSE 0 END AS conoce_ley_045,
        COALESCE(NULLIF(TRIM(CAST(estrato AS VARCHAR)), 'nan'), 'Sin_Estrato') AS estrato
    FROM '{RUTA_PARQUET}'
    WHERE Percepcion_Discriminacion IS NOT NULL
""").df()

# Si existen variables de Personas en el Parquet (sexo, edad, idioma, nivel educativo), las incorporamos automáticamente
cols_parquet = con.execute(f"DESCRIBE SELECT * FROM '{RUTA_PARQUET}'").df()['column_name'].tolist()
extras_personas = [c for c in ['s01a_02', 's01a_03', 's01a_09', 'niv_ed', 'sexo', 'edad'] if c in cols_parquet]
if extras_personas:
    df_extra = con.execute(f"SELECT {', '.join(extras_personas)} FROM '{RUTA_PARQUET}' WHERE Percepcion_Discriminacion IS NOT NULL").df()
    df_model = pd.concat([df_model, df_extra], axis=1)

# Codificación One-Hot limpia y sanitización de nombres de columnas para LightGBM/SHAP
X_raw = df_model.drop(columns=['target', 'ponderador'])
y = df_model['target'].astype(int)
w = df_model['ponderador'].astype(float)

X = pd.get_dummies(X_raw, drop_first=True, dtype=int)
X.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', c) for c in X.columns]

print(f"Dataset preparado: {X.shape[0]:,} registros | {X.shape[1]} variables predictoras | Tasa positiva: {y.mean()*100:.2f}%")

# ==============================================================================
# 2. ENTRENAMIENTO CON VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLD CV)
# ==============================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=15,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=250, learning_rate=0.05, num_leaves=31,
        is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1
    )
}

resultados_cv = []
oof_probas = {nombre: np.zeros(len(y)) for nombre in modelos}

for nombre, modelo in modelos.items():
    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr, w_tr = X.iloc[train_idx], y.iloc[train_idx], w.iloc[train_idx]
        X_va, y_va, w_va = X.iloc[val_idx], y.iloc[val_idx], w.iloc[val_idx]

        modelo.fit(X_tr, y_tr, sample_weight=w_tr)
        prob_va = modelo.predict_proba(X_va)[:, 1]
        oof_probas[nombre][val_idx] = prob_va

        # Búsqueda de umbral óptimo F1 en validación para clases desbalanceadas
        precisions, recalls, thresholds = precision_recall_curve(y_va, prob_va, sample_weight=w_va)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
        best_thresh = thresholds[np.argmax(f1_scores)] if len(thresholds) > 0 else 0.5
        pred_va = (prob_va >= best_thresh).astype(int)

        fold_metrics.append({
            "AUC_ROC": roc_auc_score(y_va, prob_va, sample_weight=w_va),
            "PR_AUC": average_precision_score(y_va, prob_va, sample_weight=w_va),
            "F1_Score": f1_score(y_va, pred_va, sample_weight=w_va),
            "Precision": precision_score(y_va, pred_va, sample_weight=w_va, zero_division=0),
            "Recall": recall_score(y_va, pred_va, sample_weight=w_va, zero_division=0),
            "Umbral_Optimo": best_thresh
        })

    df_m = pd.DataFrame(fold_metrics).mean()
    resultados_cv.append({
        "Modelo": nombre,
        "AUC-ROC (Media CV)": round(df_m["AUC_ROC"], 4),
        "PR-AUC (Media CV)": round(df_m["PR_AUC"], 4),
        "F1-Score (Media CV)": round(df_m["F1_Score"], 4),
        "Precision": round(df_m["Precision"], 4),
        "Recall": round(df_m["Recall"], 4),
        "Umbral Óptimo": round(df_m["Umbral_Optimo"], 3)
    })

df_comparativa = pd.DataFrame(resultados_cv)
print("\n=== EVALUACIÓN COMPARATIVA (VALIDACIÓN CRUZADA ESTRATIFICADA 5-FOLD) ===")
print(df_comparativa.to_string(index=False))

# ==============================================================================
# 3. EVALUACIÓN RIGUROSA: CURVAS ROC / PR Y MATRIZ DE CONFUSIÓN (MEJOR MODELO)
# ==============================================================================
mejor_modelo_nombre = df_comparativa.sort_values("AUC-ROC (Media CV)", ascending=False).iloc[0]["Modelo"]
umbral_final = df_comparativa.loc[df_comparativa["Modelo"] == mejor_modelo_nombre, "Umbral Óptimo"].values[0]
probs_mejor = oof_probas[mejor_modelo_nombre]
preds_mejor = (probs_mejor >= umbral_final).astype(int)

# 3A. Matriz de Confusión Interactiva
cm = confusion_matrix(y, preds_mejor)
cm_norm = confusion_matrix(y, preds_mejor, normalize='true') * 100

labels_cm = [[f"{cm[i,j]:,}<br>({cm_norm[i,j]:.1f}%)" for j in range(2)] for i in range(2)]
fig_cm = go.Figure(data=go.Heatmap(
    z=cm_norm,
    x=["Predicho: No Discriminado (0)", "Predicho: Discriminado (1)"],
    y=["Real: No Discriminado (0)", "Real: Discriminado (1)"],
    text=labels_cm,
    texttemplate="%{text}",
    colorscale="Blues",
    showscale=True
))
fig_cm.update_layout(
    title=f"Matriz de Confusión Out-of-Fold: {mejor_modelo_nombre} (Umbral={umbral_final})",
    xaxis_title="Predicción del Modelo",
    yaxis_title="Clase Observada (EH 2025)"
)
fig_cm.write_html("outputs/eval_matriz_confusion.html")
fig_cm.show()

# 3B. Comparación de Curvas ROC y Precision-Recall
fig_roc = go.Figure()
for nombre in modelos:
    fpr, tpr, _ = roc_curve(y, oof_probas[nombre], sample_weight=w)
    auc_val = roc_auc_score(y, oof_probas[nombre], sample_weight=w)
    fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f"{nombre} (AUC = {auc_val:.4f})"))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash', color='gray'), name="Azar (0.50)"))
fig_roc.update_layout(
    title="Comparación de Curvas ROC Ponderadas (Validación Cruzada Estratificada)",
    xaxis_title="Tasa de Falsos Positivos (1 - Especificidad)",
    yaxis_title="Tasa de Verdaderos Positivos (Sensibilidad / Recall)"
)
fig_roc.write_html("outputs/eval_curvas_roc.html")
fig_roc.show()

# ==============================================================================
# 4. INTERPRETABILIDAD AVANZADA (XAI) CON VALORES SHAP
# ==============================================================================
# Entrenamos el modelo LightGBM final sobre todo el conjunto para extraer valores SHAP exactos
modelo_xai = modelos["LightGBM"]
modelo_xai.fit(X, y, sample_weight=w)

# Muestra estratificada de 2,500 observaciones para cálculo rápido y representativo de SHAP
X_shap = X.sample(n=min(2500, len(X)), random_state=42)
explainer = shap.TreeExplainer(modelo_xai)
shap_values = explainer.shap_values(X_shap)

# Compatibilidad según versión de SHAP (lista de 2 arrays o matriz 2D directa)
if isinstance(shap_values, list):
    sv_clase1 = shap_values[1]
else:
    sv_clase1 = shap_values

# Calculamos importancia global media (|SHAP|) y dirección promedio del efecto (correlación feature vs SHAP)
importancia_shap = pd.DataFrame({
    "Variable": X.columns,
    "Importancia_SHAP_Media": np.abs(sv_clase1).mean(axis=0),
    "Direccion_Efecto": [
        np.corrcoef(X_shap[col], sv_clase1[:, idx])[0, 1] if X_shap[col].std() > 0 else 0
        for idx, col in enumerate(X.columns)
    ]
}).sort_values("Importancia_SHAP_Media", ascending=False).head(15)

importancia_shap["Impacto_en_Discriminacion"] = np.where(
    importancia_shap["Direccion_Efecto"] > 0,
    "Aumenta probabilidad (+)",
    "Reduce probabilidad (-)"
)

print("\n=== TOP 15 FACTORES EXPLICATIVOS SEGÚN VALORES SHAP (XAI) ===")
print(importancia_shap[["Variable", "Importancia_SHAP_Media", "Impacto_en_Discriminacion"]].to_string(index=False))

# Gráfico Interactivo de Importancia SHAP en Plotly
fig_shap = px.bar(
    importancia_shap.iloc[::-1],
    x="Importancia_SHAP_Media",
    y="Variable",
    color="Impacto_en_Discriminacion",
    orientation="h",
    color_discrete_map={
        "Aumenta probabilidad (+)": "#d95f02",
        "Reduce probabilidad (-)": "#1b9e77"
    },
    title="Explicabilidad del Modelo (SHAP): Principales Determinantes de la Discriminación en Bolivia",
    labels={
        "Importancia_SHAP_Media": "Impacto Medio Absoluto en la Predicción (|Valor SHAP|)",
        "Variable": "Factor Sociodemográfico / Geográfico"
    }
)
fig_shap.write_html("outputs/xai_importancia_shap.html")
fig_shap.show()
print("[OK] Archivos de evaluación y explicabilidad guardados en outputs/.")

<>:81: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:81: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_10844\4294709585.py:81: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  modelos = {


Dataset preparado: 12,358 registros | 81 variables predictoras | Tasa positiva: 14.77%

=== EVALUACIÓN COMPARATIVA (VALIDACIÓN CRUZADA ESTRATIFICADA 5-FOLD) ===
       Modelo  AUC-ROC (Media CV)  PR-AUC (Media CV)  F1-Score (Media CV)  Precision  Recall  Umbral Óptimo
Random Forest              0.6222             0.2197               0.3145     0.2377  0.4717          0.513
     LightGBM              0.6089             0.2090               0.3076     0.2317  0.4588          0.560


c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:3038: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



=== TOP 15 FACTORES EXPLICATIVOS SEGÚN VALORES SHAP (XAI) ===
               Variable  Importancia_SHAP_Media Impacto_en_Discriminacion
    departamento_La_Paz                0.300990  Aumenta probabilidad (+)
    percibe_inseguridad                0.152031  Aumenta probabilidad (+)
     departamento_Pando                0.040377  Aumenta probabilidad (+)
departamento_Santa_Cruz                0.038295   Reduce probabilidad (-)
    departamento_Tarija                0.037560   Reduce probabilidad (-)
departamento_Chuquisaca                0.026939  Aumenta probabilidad (+)
departamento_Cochabamba                0.025071  Aumenta probabilidad (+)
     departamento_Oruro                0.023249  Aumenta probabilidad (+)
        area_geo_Urbana                0.021995   Reduce probabilidad (-)
            estrato_213                0.018533  Aumenta probabilidad (+)
            estrato_314                0.016074   Reduce probabilidad (-)
            estrato_713                0.015635  

[OK] Archivos de evaluación y explicabilidad guardados en outputs/.
